# Reproduce published methods (Progress Review 2)

First runs by Phúc; this clean version is run top to bottom in one Colab session.

1. **Official RGT on the paper's splits** (random 7:2:1, seeds 1–5, follower + friend). Checks the published 92.1 accuracy / 90.4 macro-F1.
2. **Official RGT model on our setup:** our `split()`, our 6 cleaned relations, 5 seeds. This is the strong baseline the professor asked for.

BotHP runs in `bothp.ipynb` (separate runtime, Python 3.10) and writes `results/bothp.csv`.

**Rules:** step 1 uses the paper's random splits, which include our test accounts, so it only checks published numbers and never tunes our models. Step 2 reports **validation only**; the test set stays locked.

**Before you run:** Runtime > Change runtime type > GPU (L4 or A100 if possible; a T4 may run out of memory in step 2). Then Runtime > Run all, and keep the outputs. At the end, download the zip and the notebook (File > Download > .ipynb) and commit both.

In [ ]:
import sys, subprocess, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Turn on a GPU: Runtime > Change runtime type"

In [ ]:
# Team repo, environment and data (same steps as notebooks/template.ipynb)
import os, glob, shutil, subprocess, zipfile
import numpy as np, pandas as pd
TEAM = "/content/sna-content-moderation"
%cd /content
!test -d sna-content-moderation || git clone -q https://github.com/ThSonDev/sna-content-moderation.git
%cd /content/sna-content-moderation
!git pull -q
!pip install -q gdown -r requirements.txt
!python install_pyg.py
if not os.path.exists("data/MGTAB/features.pt"):
    subprocess.run(["gdown", "1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe", "-O", "/content/MGTAB.zip"], check=True)
    zipfile.ZipFile("/content/MGTAB.zip").extractall("/content/_mgtab")
    hit = glob.glob("/content/_mgtab/**/features.pt", recursive=True)
    assert hit, "features.pt not found in the zip"
    os.makedirs("data", exist_ok=True)
    shutil.move(os.path.dirname(hit[0]), "data/MGTAB")
!python mgtab.py
os.makedirs("results/logs", exist_ok=True)
RESULTS = []

## 1. Official RGT on the paper's splits

`RGT.py` from GraphDetec/MGTAB with its defaults (seeds 1–5, hidden 128, dropout 0.5, lr 1e-3, weight decay 5e-3, 200 epochs, relations 0 and 1). It reports test results at the epoch with the best validation accuracy, and macro-F1.

Colab patches only, no model change: GPU id 8 → 0, `.item()` → `float()` (new scikit-learn returns plain floats), `@torch.no_grad()` on `test()` (saves memory).

In [ ]:
%cd /content
!test -d MGTAB || git clone -q https://github.com/GraphDetec/MGTAB.git
%cd /content/MGTAB
!git log -1 --format="MGTAB repo commit %h (%ad)"
shutil.rmtree("Dataset/MGTAB", ignore_errors=True)
if os.path.islink("Dataset/MGTAB") or os.path.isfile("Dataset/MGTAB"):
    os.remove("Dataset/MGTAB")
os.makedirs("Dataset", exist_ok=True)
shutil.copytree(f"{TEAM}/data/MGTAB", "Dataset/MGTAB")
!sed -i 's/CUDA_VISIBLE_DEVICES"\] = "8"/CUDA_VISIBLE_DEVICES"] = "0"/' RGT.py
!sed -i -E 's/(acc_[a-z]+)\.item\(\)/float(\1)/g' RGT.py
!grep -q "@torch.no_grad()" RGT.py || sed -i 's/^    def test():/    @torch.no_grad()\n    def test():/' RGT.py
!grep -n "CUDA_VISIBLE\|float(acc\|no_grad\|average=" RGT.py

In [ ]:
# 5 seeds, about 11 minutes on an L4. The full log is saved to results/logs/.
import re
%cd /content/MGTAB
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python RGT.py > rgt_paper_split.log 2>&1; grep -v "^Epoch" rgt_paper_split.log | tail -15
log = open("/content/MGTAB/rgt_paper_split.log").read()
runs = np.array(re.findall(r"test_accuracy= ([\d.]+) precision= ([\d.]+) recall= ([\d.]+) f1_score= ([\d.]+)", log), dtype=float)
assert len(runs) == 5, f"expected 5 seeds, found {len(runs)}; check the log"
shutil.copy("/content/MGTAB/rgt_paper_split.log", f"{TEAM}/results/logs/rgt_paper_split.log")
acc, f1 = runs[:, 0], runs[:, 3]
print("per seed accuracy", acc, "macro-F1", f1)
RESULTS.append(dict(method="RGT (official script)", split="paper: random 7:2:1, seeds 1-5", relations="follower, friend",
                    eval_set="paper test (10%)", runs=5, accuracy=acc.mean(), accuracy_std=acc.std(),
                    macro_f1=f1.mean(), macro_f1_std=f1.std(),
                    note="Published 92.1 / 90.4. Epoch chosen by validation accuracy. Colab patches only."))
pd.DataFrame(RESULTS)

## 2. Official RGT model on our setup

The `RGT` class from the official `models.py`, with the official hyperparameters, loss (plain cross-entropy) and optimizer (AdamW). What changes:

- **Data:** our `split()` (7,139 / 1,530 / 1,530) and our 6 cleaned relations from `relations()` (no self-loops or duplicates; URL and hashtag in both directions).
- **Epoch choice:** best validation accuracy, as in the official script, but measured in eval mode (the official script measures it with dropout on).
- **Metrics:** our `evaluate()` on validation, with the threshold from `best_threshold`, so the numbers compare directly with `results/baselines.csv`.
- 5 seeds (model initialisation only; the split is fixed).

In [ ]:
import argparse, torch.nn.functional as F
sys.path.insert(0, "/content/MGTAB")
sys.path.insert(0, TEAM)
%cd /content/sna-content-moderation
from models import RGT                      # official model code
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS

X, y, _, E, T, _ = load()
rel = relations(E, T, len(y))
train, val, test = split(y)                 # test stays locked
src, dst, typ = [], [], []
for k, r in enumerate(RELATIONS):
    s, d = rel[r]
    if r in ("url", "hashtag"):             # undirected, stored once: add the reverse direction
        s, d = np.r_[s, d], np.r_[d, s]
    src.append(s); dst.append(d); typ.append(np.full(len(s), k))

dev = torch.device("cuda")
x = torch.tensor(X, dtype=torch.float32, device=dev)
yt = torch.tensor(y, dtype=torch.long, device=dev)
ei = torch.tensor(np.vstack([np.concatenate(src), np.concatenate(dst)]), dtype=torch.long, device=dev)
et = torch.tensor(np.concatenate(typ), dtype=torch.long, device=dev)
tr = torch.tensor(train, device=dev)
print("edges", tuple(ei.shape), {r: int((et == k).sum()) for k, r in enumerate(RELATIONS)})

args = argparse.Namespace(relation_select=list(range(len(RELATIONS))), features_num=X.shape[1], out_dim=2,
                          hidden_dimension=128, out_channel=64, trans_head=4, semantic_head=4, dropout=0.5)
LR, WEIGHT_DECAY, EPOCHS = 1e-3, 5e-3, 200  # official defaults

def run_rgt(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = RGT(args).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_acc, best_prob, best_epoch = -1, None, 0
    for epoch in range(EPOCHS):
        model.train(); opt.zero_grad()
        F.cross_entropy(model(x, ei, et)[tr], yt[tr]).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            prob = F.softmax(model(x, ei, et), 1)[:, 1].cpu().numpy()
        acc = ((prob[val] >= 0.5) == y[val]).mean()
        if acc > best_acc:
            best_acc, best_prob, best_epoch = acc, prob, epoch + 1
    m = evaluate(y[val], best_prob[val], best_threshold(y[val], best_prob[val]))
    return {"seed": seed, "best_epoch": best_epoch, **m}

per_seed = pd.DataFrame([run_rgt(s) for s in [1, 2, 3, 4, 5]])
per_seed.to_csv("results/logs/rgt_our_split_per_seed.csv", index=False)
display(per_seed.round(4))
mean, std = per_seed.mean(), per_seed.std(ddof=0)
RESULTS.append(dict(method="RGT (official model)", split="ours: split(), seed 42", relations="6 cleaned",
                    eval_set="our validation (15%)", runs=5, accuracy=mean["accuracy"], accuracy_std=std["accuracy"],
                    macro_f1=mean["macro_f1"], macro_f1_std=std["macro_f1"], pr_auc=mean["pr_auc"],
                    pr_auc_std=std["pr_auc"], p_at_100=mean["p@100"], p_at_200=mean["p@200"],
                    note="Official hyperparameters. Epoch by validation accuracy; threshold tuned on validation."))

## 3. Save and download

`results/reproduce.csv` plus the logs, as one zip. Also save this notebook with its outputs (File > Download > .ipynb).

In [ ]:
%cd /content/sna-content-moderation
df = pd.DataFrame(RESULTS)
df.to_csv("results/reproduce.csv", index=False)
display(df.round(4))
!zip -q -r /content/reproduce_outputs.zip results/reproduce.csv results/logs
from google.colab import files
files.download("/content/reproduce_outputs.zip")